## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

ZIP_PATH = '/content/drive/MyDrive/Advanced_CV_Project/openfake_subset.zip'  # <-- change if needed
assert os.path.exists(ZIP_PATH), f'Zip not found at {ZIP_PATH}'
print(f'Found zip: {ZIP_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found zip: /content/drive/MyDrive/Advanced_CV_Project/openfake_subset.zip


## Cell 2 — Extract Dataset

In [ ]:
import zipfile, os

EXTRACT_DIR = '/content/dataset'
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting zip (this may take a few minutes)...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)
print('Done.')


for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        print(f'{indent}  ... ({len(files)} files)')

Extracting zip (this may take a few minutes)...
Done.
dataset/
  images/


## Cell 3 — Load Metadata and Fix Paths

In [ ]:
import pandas as pd

METADATA_PATH = '/content/dataset/metadata.csv'
df = pd.read_csv(METADATA_PATH)


df['image_path'] = df['image_path'].str.replace(
    '/content/openfake_subset/images',
    '/content/dataset/images',
    regex=False
)

sample = df['image_path'].iloc[0]
print(f'Remapped path sample : {sample}')
print(f'File exists          : {os.path.exists(sample)}')
print(f'\nShape: {df.shape}')
print(f'\nLabels:\n{df["label"].value_counts()}')

Remapped path sample : /content/dataset/images/0.jpg
File exists          : True

Shape: (8989, 4)

Labels:
label
real                3000
stable_diffusion    3000
midjourney          2989
Name: count, dtype: int64


## Cell 4 — Create Train / Val / Test Splits

**Same seed (42) and same 80/10/10 split** — this guarantees the test set is identical across models

In [ ]:
import pandas as pd, os, shutil

METADATA_PATH = '/content/dataset/metadata.csv'
OUT_BASE      = '/content/splits'

df = pd.read_csv(METADATA_PATH)
df['image_path'] = df['image_path'].str.replace(
    '/content/openfake_subset/images',
    '/content/dataset/images', regex=False
)

sample = df['image_path'].iloc[0]
print(f'Remapped path sample : {sample}')
print(f'File exists          : {os.path.exists(sample)}')

df['split_label'] = df['label'].apply(lambda x: 'real' if str(x).lower() == 'real' else 'fake')

real_df = df[df['split_label'] == 'real'].copy()
fake_df = df[df['split_label'] == 'fake'].copy()

print(f'\nReal: {len(real_df)}  Fake: {len(fake_df)}')

def split_df(dataframe, splits=(0.8, 0.1, 0.1)):
    dataframe = dataframe.sample(frac=1, random_state=42).reset_index(drop=True)
    n = len(dataframe)
    n_train = int(n * splits[0])
    n_val   = int(n * splits[1])
    return {
        'train': dataframe.iloc[:n_train],
        'val'  : dataframe.iloc[n_train:n_train+n_val],
        'test' : dataframe.iloc[n_train+n_val:]
    }

def copy_split(split_dict, label, out_base):
    for split_name, sdf in split_dict.items():
        dst = os.path.join(out_base, split_name, label)
        os.makedirs(dst, exist_ok=True)
        copied, missing = 0, 0
        for _, row in sdf.iterrows():
            src = row['image_path']
            if not os.path.exists(src):
                missing += 1
            else:
                shutil.copy(src, os.path.join(dst, os.path.basename(src)))
                copied += 1
        print(f'  {split_name}/{label}: {copied} copied, {missing} missing')

print('\nCopying real images...')
copy_split(split_df(real_df), 'real', OUT_BASE)
print('Copying fake images...')
copy_split(split_df(fake_df), 'fake', OUT_BASE)

print('\nFinal split summary:')
for split_name in ['train', 'val', 'test']:
    r = len(os.listdir(f'{OUT_BASE}/{split_name}/real'))
    f = len(os.listdir(f'{OUT_BASE}/{split_name}/fake'))
    print(f'  {split_name:5s} → real: {r:4d}  fake: {f:4d}  total: {r+f}')

# Save split metadata to Drive so it can be reused
all_splits = []
for split_name in ['train', 'val', 'test']:
    for label in ['real', 'fake']:
        folder = f'{OUT_BASE}/{split_name}/{label}'
        for fname in os.listdir(folder):
            all_splits.append({'filename': fname, 'split': split_name, 'label': label})
pd.DataFrame(all_splits).to_csv('/content/drive/MyDrive/Advanced_CV_Project/model_split_metadata.csv', index=False)
print('\nSplit metadata saved to Drive.')

Remapped path sample : /content/dataset/images/0.jpg
File exists          : True

Real: 3000  Fake: 5989

Copying real images...
  train/real: 2400 copied, 0 missing
  val/real: 300 copied, 0 missing
  test/real: 300 copied, 0 missing
Copying fake images...
  train/fake: 4791 copied, 0 missing
  val/fake: 598 copied, 0 missing
  test/fake: 600 copied, 0 missing

Final split summary:
  train → real: 2400  fake: 4791  total: 7191
  val   → real:  300  fake:  598  total: 898
  test  → real:  300  fake:  600  total: 900

Split metadata saved to Drive.
